In [ ]:
import os

import pandas as pd
from scipy import stats
from tqdm import tqdm

from pratvi_strategy import Strategy

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


def run_backtest(df, strategy_class, *args, **kwargs):
    results = []

    for side in ["BUY", "SELL"]:
        strat = strategy_class(side, *args, **kwargs)
        for minute, grp in tqdm(df.groupby("Minute")):
            twap_price = (
                grp["AskPrice_1"].iloc[0]
                if side == "BUY"
                else grp["BidPrice_1"].iloc[0]
            )

            exec_price = None
            exec_time = None

            executed = False
            for idx, row in grp.iterrows():
                if strat.on_tick(row, idx):
                    if not executed:
                        exec_price = (
                            row["AskPrice_1"] if side == "BUY" else row["BidPrice_1"]
                        )
                        exec_time = idx
                        executed = True
                    else:
                        break # Only consider the one execution 

            if exec_price is None:
                last_row = grp.iloc[-1]
                exec_price = (
                    last_row["AskPrice_1"] if side == "BUY" else last_row["BidPrice_1"]
                )
                exec_time = grp.index[-1]

            improvement = (
                twap_price - exec_price if side == "BUY" else exec_price - twap_price
            )

            optm_price = (
                grp["AskPrice_1"].min() if side == "BUY" else grp["BidPrice_1"].max()
            )

            results.append(
                {
                    "Minute": minute,
                    "Side": side,
                    "TWAP_Price": twap_price,
                    "Exec_Price": exec_price,
                    "Optm_Price": optm_price,
                    "Improvement_bps": improvement / twap_price * 10000,
                    "Optm_Improvement_bps": abs(optm_price - twap_price)
                    / twap_price
                    * 10000,
                    "Exec_Time": exec_time,
                }
            )

    return pd.DataFrame(results)


In [ ]:

tickers = ["AMZN", "GOOG", "INTC", "MSFT"]

summary_data = []
for ticker in tickers:
    file_path = f"data/{ticker}_5levels_train.csv"

    if not os.path.exists(file_path):
        print(f"File '{file_path}' doesn't exist.")
        continue

    print(f"Backtesting {ticker} ...")

    df = pd.read_csv(file_path)

    df.columns = [c.strip() for c in df.columns]
    df["Time_dt"] = pd.to_datetime(
        df["Time"], format="%H:%M:%S.%f", errors="coerce"
    )
    df = df.sort_values("Time_dt")
    df.set_index("Time_dt", inplace=True)
    df["Minute"] = df.index.floor("min")
    total_size = df["BidSize_1"] + df["AskSize_1"]
    
    # Microprice
    df['Microprice'] = (
        df["BidPrice_1"] * df["AskSize_1"] +
        df["AskPrice_1"] * df["BidSize_1"]
    ) / total_size.replace(0, 1)
    df['dSpread'] = df["Spread"].diff().shift(1)

    # TFI
    mask = (df["VisibleExecution_1=Yes_0=No"] == 1) | (df["HiddenExecution_1=Yes_0=No"] == 1)
    df["TFI"] = 0.0
    df.loc[mask, "TFI"] = (
        df.loc[mask, "Direction_1=Buy_-1=Sell"] * df.loc[mask, "Size"]
        / (df.loc[mask, "BidSize_1"] + df.loc[mask, "AskSize_1"])
    )
    df['Rolling_TFI'] = df['TFI'].rolling(window=100, min_periods=1).mean().shift(1).fillna(0)
    df['Rolling_std_TFI'] = df['TFI'].rolling(window=100, min_periods=1).std().shift(1).fillna(0)

    result_df = run_backtest(df, Strategy)

    output_name = f"results/{ticker}_backtest_result.csv"
    result_df.to_csv(output_name, index=False)

    print(f">>> {ticker} Results saved to {output_name}.")

    for side in ["BUY", "SELL"]:
        side_df = result_df[result_df["Side"] == side]
        imp_series = side_df["Improvement_bps"]

        if len(imp_series) > 1:
            mean_imp = imp_series.mean()
            std_imp = imp_series.std()
            t_stat, p_val = stats.ttest_1samp(imp_series, 0)
        else:
            mean_imp, std_imp, t_stat, p_val = 0.0, 0.0, 0.0, 1.0

        print(
            f"    [{side:4s}] Mean: {mean_imp:>7.4f} bps | Std: {std_imp:>6.4f} | T-Stat: {t_stat:>7.4f} (p-val: {p_val:>6.4f})"
        )

        summary_data.append(
            {
                "Ticker": ticker,
                "Side": side,
                "Mean_Improvement": mean_imp,
                "Std_Improvement": std_imp,
                "T_Statistic": t_stat,
                "P_Value": p_val,
            }
        )

    total_buy_exec = result_df.loc[result_df["Side"] == "BUY", "Exec_Price"].sum()
    total_sell_exec = result_df.loc[result_df["Side"] == "SELL", "Exec_Price"].sum()
    total_buy_twap = result_df.loc[result_df["Side"] == "BUY", "TWAP_Price"].sum()
    total_sell_twap = result_df.loc[result_df["Side"] == "SELL", "TWAP_Price"].sum()
    pct_improv = 100 - 100 * (total_buy_exec - total_sell_exec) / (
        total_buy_twap - total_sell_twap
    )
    print(f"Percentage Improvement: {pct_improv:>7.4f}%")
    print("-" * 70)

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv("results/summary_stats.csv", index=False)
    print("\nAll backtests completed. Summary saved to results/summary_stats.csv")

Backtesting AMZN ...


100%|██████████| 270/270 [00:02<00:00, 105.14it/s]


>>> AMZN Results saved to results/AMZN_backtest_result.csv.
    [BUY ] Mean:  2.9600 bps | Std: 4.9734 | T-Stat:  9.7793 (p-val: 0.0000)
    [SELL] Mean:  1.5936 bps | Std: 4.8453 | T-Stat:  5.4043 (p-val: 0.0000)
Percentage Improvement: 73.1240%
----------------------------------------------------------------------
Backtesting GOOG ...


100%|██████████| 270/270 [00:01<00:00, 167.06it/s]


>>> GOOG Results saved to results/GOOG_backtest_result.csv.
    [BUY ] Mean:  2.0321 bps | Std: 3.9410 | T-Stat:  8.4727 (p-val: 0.0000)
    [SELL] Mean:  1.3960 bps | Std: 3.0546 | T-Stat:  7.5094 (p-val: 0.0000)
Percentage Improvement: 66.6792%
----------------------------------------------------------------------
Backtesting INTC ...


100%|██████████| 270/270 [00:04<00:00, 61.10it/s]


>>> INTC Results saved to results/INTC_backtest_result.csv.
    [BUY ] Mean:  1.6897 bps | Std: 2.8643 | T-Stat:  9.6934 (p-val: 0.0000)
    [SELL] Mean:  1.7310 bps | Std: 2.6981 | T-Stat: 10.5421 (p-val: 0.0000)
Percentage Improvement: 90.9420%
----------------------------------------------------------------------
Backtesting MSFT ...


100%|██████████| 270/270 [00:03<00:00, 72.63it/s] 

>>> MSFT Results saved to results/MSFT_backtest_result.csv.
    [BUY ] Mean:  1.8712 bps | Std: 2.8361 | T-Stat: 10.8413 (p-val: 0.0000)
    [SELL] Mean:  1.4860 bps | Std: 2.3074 | T-Stat: 10.5820 (p-val: 0.0000)
Percentage Improvement: 101.0909%
----------------------------------------------------------------------

All backtests completed. Summary saved to results/summary_stats.csv


In [2]:
# Zijie Strategy Backtest Script
tickers = ["AMZN", "GOOG", "INTC", "MSFT"]

summary_data = []
for ticker in tickers:
    file_path = f"data/{ticker}_5levels_train.csv"

    if not os.path.exists(file_path):
        print(f"File '{file_path}' doesn't exist.")
        continue

    print(f"Backtesting {ticker} ...")

    df = pd.read_csv(file_path)

    df.columns = [c.strip() for c in df.columns]
    df["Time_dt"] = pd.to_datetime(
        df["Time"], format="%H:%M:%S.%f", errors="coerce"
    )
    df = df.sort_values("Time_dt")
    df.set_index("Time_dt", inplace=True)
    df["Minute"] = df.index.floor("min")
    total_size = df["BidSize_1"] + df["AskSize_1"]
    df['Microprice'] = (
        df["BidPrice_1"] * df["AskSize_1"] +
        df["AskPrice_1"] * df["BidSize_1"]
    ) / total_size.replace(0, 1)
    result_df = run_backtest(df, Strategy)

    output_name = f"results/{ticker}_backtest_result.csv"
    result_df.to_csv(output_name, index=False)

    print(f">>> {ticker} Results saved to {output_name}.")

    for side in ["BUY", "SELL"]:
        side_df = result_df[result_df["Side"] == side]
        imp_series = side_df["Improvement_bps"]

        if len(imp_series) > 1:
            mean_imp = imp_series.mean()
            std_imp = imp_series.std()
            t_stat, p_val = stats.ttest_1samp(imp_series, 0)
        else:
            mean_imp, std_imp, t_stat, p_val = 0.0, 0.0, 0.0, 1.0

        print(
            f"    [{side:4s}] Mean: {mean_imp:>7.4f} bps | Std: {std_imp:>6.4f} | T-Stat: {t_stat:>7.4f} (p-val: {p_val:>6.4f})"
        )

        summary_data.append(
            {
                "Ticker": ticker,
                "Side": side,
                "Mean_Improvement": mean_imp,
                "Std_Improvement": std_imp,
                "T_Statistic": t_stat,
                "P_Value": p_val,
            }
        )

    total_buy_exec = result_df.loc[result_df["Side"] == "BUY", "Exec_Price"].sum()
    total_sell_exec = result_df.loc[result_df["Side"] == "SELL", "Exec_Price"].sum()
    total_buy_twap = result_df.loc[result_df["Side"] == "BUY", "TWAP_Price"].sum()
    total_sell_twap = result_df.loc[result_df["Side"] == "SELL", "TWAP_Price"].sum()
    pct_improv = 100 - 100 * (total_buy_exec - total_sell_exec) / (
        total_buy_twap - total_sell_twap
    )
    print(f"Percentage Improvement: {pct_improv:>7.4f}%")
    print("-" * 70)

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv("results/summary_stats.csv", index=False)
    print("\nAll backtests completed. Summary saved to results/summary_stats.csv")

Backtesting AMZN ...


100%|██████████| 270/270 [00:01<00:00, 209.82it/s]


>>> AMZN Results saved to results/AMZN_backtest_result.csv.
    [BUY ] Mean:  0.7515 bps | Std: 3.6407 | T-Stat:  3.3919 (p-val: 0.0008)
    [SELL] Mean:  0.7607 bps | Std: 3.9665 | T-Stat:  3.1512 (p-val: 0.0018)
Percentage Improvement: 24.2682%
----------------------------------------------------------------------
Backtesting GOOG ...


100%|██████████| 270/270 [00:00<00:00, 389.15it/s]


>>> GOOG Results saved to results/GOOG_backtest_result.csv.
    [BUY ] Mean:  0.9949 bps | Std: 3.7507 | T-Stat:  4.3585 (p-val: 0.0000)
    [SELL] Mean:  0.4555 bps | Std: 2.1646 | T-Stat:  3.4574 (p-val: 0.0006)
Percentage Improvement: 28.2177%
----------------------------------------------------------------------
Backtesting INTC ...


100%|██████████| 270/270 [00:03<00:00, 69.33it/s]


>>> INTC Results saved to results/INTC_backtest_result.csv.
    [BUY ] Mean:  1.7028 bps | Std: 2.7187 | T-Stat: 10.2918 (p-val: 0.0000)
    [SELL] Mean:  1.6363 bps | Std: 2.4413 | T-Stat: 11.0134 (p-val: 0.0000)
Percentage Improvement: 88.7681%
----------------------------------------------------------------------
Backtesting MSFT ...


100%|██████████| 270/270 [00:02<00:00, 96.59it/s] 


>>> MSFT Results saved to results/MSFT_backtest_result.csv.
    [BUY ] Mean:  1.8339 bps | Std: 2.6811 | T-Stat: 11.2397 (p-val: 0.0000)
    [SELL] Mean:  1.4487 bps | Std: 2.2842 | T-Stat: 10.4212 (p-val: 0.0000)
Percentage Improvement: 98.9091%
----------------------------------------------------------------------

All backtests completed. Summary saved to results/summary_stats.csv
